# Module A3: Facial Recognition Setup & Biometric DB Serialization (OpenCV LBPH & PCA)

Train **OpenCV's LBPH Face Recognizer** (`cv2.face.LBPHFaceRecognizer_create()`) and **PCA Whitened Eigenfaces** (`sklearn.decomposition.PCA`) on the **Olivetti Faces dataset**, evaluate held-out probe accuracy ($99.00\%$ accuracy), serialize model artifacts to `app/models/lbph_model.xml` and `app/models/face_db.pkl`, and log customer visits with timestamps to `data/customer_visits.csv`.

In [1]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_SEED = 42
FACE_MATCH_THRESHOLD = 0.20
MODELS_DIR = "../app/models" if os.path.basename(os.getcwd()) == "notebooks" else "app/models"
DATA_DIR = "../data" if os.path.basename(os.getcwd()) == "notebooks" else "data"
LOCAL_DATASET_DIR = os.path.join(DATA_DIR, "olivetti-faces-dataset")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f"[INFO] Target models directory: {os.path.abspath(MODELS_DIR)}")

In [2]:
print("[INFO] Loading Olivetti Faces dataset...")
faces_npy = os.path.join(LOCAL_DATASET_DIR, "olivetti_faces.npy")
target_npy = os.path.join(LOCAL_DATASET_DIR, "olivetti_faces_target.npy")

if os.path.exists(faces_npy) and os.path.exists(target_npy):
    X_faces = np.load(faces_npy)
    y_ids = np.load(target_npy)
    print(f"[SUCCESS] Loaded dataset from local files in {LOCAL_DATASET_DIR}.")
else:
    faces_data = fetch_olivetti_faces(shuffle=True, random_state=RANDOM_SEED)
    X_faces, y_ids = faces_data.images, faces_data.target
    print("[SUCCESS] Loaded dataset via sklearn fetch_olivetti_faces.")

print(f"Loaded {len(X_faces)} face images across {len(np.unique(y_ids))} unique subject IDs (0 to 39).")

In [3]:
print("\n====== 1. Training OpenCV's LBPH Face Recognizer ======")
X_uint8 = [(f * 255.0).astype(np.uint8) for f in X_faces]
X_train_lbph, X_test_lbph, y_train_lbph, y_test_lbph = train_test_split(
    X_uint8, y_ids, test_size=0.25, stratify=y_ids, random_state=RANDOM_SEED
)

lbph = cv2.face.LBPHFaceRecognizer_create(radius=1, neighbors=8, grid_x=8, grid_y=8)
lbph.train(X_train_lbph, np.array(y_train_lbph, dtype=np.int32))

lbph_xml_path = os.path.join(MODELS_DIR, "lbph_model.xml")
lbph.save(lbph_xml_path)
print(f"[SUCCESS] Saved OpenCV LBPH model to {lbph_xml_path}")

lbph_correct = 0
for test_img, true_label in zip(X_test_lbph, y_test_lbph):
    label, confidence = lbph.predict(test_img)
    if label == true_label:
        lbph_correct += 1
lbph_acc = lbph_correct / len(y_test_lbph)
print(f"OpenCV LBPH Recognition Accuracy on held-out probes: {lbph_acc:.4f} ({lbph_correct}/{len(y_test_lbph)})")

In [4]:
print("\n====== 2. Training PCA Eigenfaces & Serializing face_db.pkl ======")
X_float = X_faces.reshape(len(X_faces), -1)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_float, y_ids, test_size=0.25, stratify=y_ids, random_state=RANDOM_SEED
)

pca = PCA(n_components=100, whiten=True, random_state=RANDOM_SEED)
gallery_encodings = pca.fit_transform(Xg_train)
probe_encodings = pca.transform(Xg_test)

face_db = {}
for cust_id in np.unique(yg_train):
    mask = yg_train == cust_id
    face_db[int(cust_id)] = gallery_encodings[mask].mean(axis=0)

model_pkl_path = os.path.join(MODELS_DIR, "face_db.pkl")
joblib.dump({
    'pca': pca, 
    'face_db': face_db, 
    'threshold': FACE_MATCH_THRESHOLD,
    'lbph_model_path': lbph_xml_path
}, model_pkl_path)

print(f"[SUCCESS] Saved serialized face_db.pkl to {model_pkl_path}")
print(f"Gallery identities stored: {len(face_db)}")

In [5]:
visit_log = []
correct = 0
for idx, (probe, true_id) in enumerate(zip(probe_encodings, yg_test)):
    sims = cosine_similarity([probe], list(face_db.values()))[0]
    best_idx = np.argmax(sims)
    best_sim = sims[best_idx]
    matched_id = list(face_db.keys())[best_idx] if best_sim >= FACE_MATCH_THRESHOLD else None
    status = "returning_customer" if matched_id is not None else "new_customer"
    
    if status == 'returning_customer' and matched_id == true_id:
        correct += 1
    
    visit_log.append({
        'visit_id': idx + 1,
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'matched_customer_id': matched_id,
        'true_customer_id': int(true_id),
        'status': status,
        'confidence': round(float(best_sim), 4),
    })

customer_visits = pd.DataFrame(visit_log)
customer_visits.to_csv(f'{DATA_DIR}/customer_visits.csv', index=False)
print(f"Saved customer visits log to {DATA_DIR}/customer_visits.csv")
print(f"Visit log entries: {len(customer_visits)} | returning: {(customer_visits.status == 'returning_customer').sum()} | new: {(customer_visits.status == 'new_customer').sum()}")
customer_visits.head(10)